# Data Preparation — Phase 3

**Source:** `milistu/AMAZON-Products-2023` (117K rows, downloaded as Arrow files)  
**Output:** `data/train.parquet`, `data/test.parquet`, `models/label_encoder.pkl`  
**Taxonomy:** UNSPSC Segment (L1)


In [ ]:
import pyarrow as pa
import pyarrow.ipc as ipc
import pandas as pd
import os

DATA_DIR  = '/home/tayana-gpu/ML/project_01_product_classifier/data'
KEEP_COLS = ['parent_asin', 'title', 'description', 'features', 'main_category']

arrow_files = sorted([
    f for f in os.listdir(DATA_DIR)
    if f.startswith('amazon_') and f.endswith('.arrow')
])

tables = []
for fname in arrow_files:
    with pa.memory_map(os.path.join(DATA_DIR, fname), 'r') as src:
        t = ipc.open_stream(src).read_all()
        tables.append(t.select(KEEP_COLS))

df = pa.concat_tables(tables).to_pandas()

print(f'Loaded: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nmain_category value counts (top 20):')
print(df['main_category'].value_counts().head(20))
print(f'\nNull main_category: {df["main_category"].isna().sum():,}')

In [ ]:
# Amazon 2023 category → UNSPSC Segment mapping
AMAZON_TO_UNSPSC = {
    'AMAZON FASHION':             'Apparel and Luggage and Personal Care Products',
    'Clothing, Shoes & Jewelry':  'Apparel and Luggage and Personal Care Products',
    'Handmade':                   'Apparel and Luggage and Personal Care Products',
    'Amazon Home':                'Domestic Appliances and Supplies and Consumer Electronic Products',
    'Home & Kitchen':             'Domestic Appliances and Supplies and Consumer Electronic Products',
    'Appliances':                 'Domestic Appliances and Supplies and Consumer Electronic Products',
    'Tools & Home Improvement':   'Building and Construction Machinery and Accessories',
    'All Electronics':            'Electronic Components and Supplies',
    'Electronics':                'Electronic Components and Supplies',
    'Cell Phones & Accessories':  'Electronic Components and Supplies',
    'Computers':                  'Electronic Components and Supplies',
    'Camera & Photo':             'Electronic Components and Supplies',
    'Car Electronics':            'Electronic Components and Supplies',
    'GPS & Navigation':           'Electronic Components and Supplies',
    'Home Audio & Theater':       'Electronic Components and Supplies',
    'Portable Audio & Accessories': 'Electronic Components and Supplies',
    'Automotive':                 'Transportation and Storage and Mail Services',
    'Sports & Outdoors':          'Sports and Recreational Equipment and Supplies',
    'Toys & Games':               'Sports and Recreational Equipment and Supplies',
    'Video Games':                'Sports and Recreational Equipment and Supplies',
    'Grocery':                    'Food Beverage and Tobacco Products',
    'Grocery & Gourmet Food':     'Food Beverage and Tobacco Products',
    'Health & Personal Care':     'Pharmaceuticals and Healthcare Products',
    'All Beauty':                 'Pharmaceuticals and Healthcare Products',
    'Beauty':                     'Pharmaceuticals and Healthcare Products',
    'Baby':                       'Pharmaceuticals and Healthcare Products',
    'Pet Supplies':               'Animals and Birds and Fish',
    'Books':                      'Printed Media',
    'Digital Music':              'Audio and Visual Presentation and Composing Equipment',
    'Musical Instruments':        'Audio and Visual Presentation and Composing Equipment',
    'Movies & TV':                'Audio and Visual Presentation and Composing Equipment',
    'Appstore for Android':       'Information Technology Broadcasting and Telecommunications',
    'Software':                   'Information Technology Broadcasting and Telecommunications',
    'Office Products':            'Office Equipment and Accessories and Supplies',
    'Arts, Crafts & Sewing':      'Office Equipment and Accessories and Supplies',
    'Collectibles & Fine Art':    'Office Equipment and Accessories and Supplies',
    'Collectible Coins':          'Office Equipment and Accessories and Supplies',
    'Industrial & Scientific':    'Industrial Machinery and Equipment',
    'Patio, Lawn & Garden':       'Farming and Fishing and Forestry and Wildlife Machinery',
}

df['unspsc_segment'] = df['main_category'].map(AMAZON_TO_UNSPSC)

unmapped = df[df['unspsc_segment'].isna()]['main_category'].value_counts()
print(f'Unmapped categories:')
print(unmapped)

df = df.dropna(subset=['unspsc_segment']).copy()
print(f'\nRows after dropping unmapped: {len(df):,}')
print(f'\nUNSPSC Segment distribution:')
print(df['unspsc_segment'].value_counts())

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s\-\/\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def join_list(field):
    if isinstance(field, list):
        return ' '.join(str(x) for x in field if x)
    if isinstance(field, str):
        return field
    return ''

df['text'] = (
    df['title'].apply(clean_text) + ' ' +
    df['description'].apply(join_list).apply(clean_text) + ' ' +
    df['features'].apply(join_list).apply(clean_text)
).str.strip()

before = len(df)
df = df[df['text'].str.split().str.len() >= 3].copy()
df = df.drop_duplicates(subset=['text']).copy()
print(f'Dropped {before - len(df):,} rows (short text or duplicates)')

counts = df['unspsc_segment'].value_counts()
valid  = counts[counts >= 100].index
dropped_segs = counts[counts < 100].index.tolist()
df = df[df['unspsc_segment'].isin(valid)].copy()

print(f'Dropped segments (< 100 rows): {dropped_segs}')
print(f'\nFinal rows: {len(df):,}')
print(f'Final segments: {df["unspsc_segment"].nunique()}')
print(f'\nFinal class distribution:')
print(df['unspsc_segment'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

PROJ = '/home/tayana-gpu/ML/project_01_product_classifier'

le = LabelEncoder()
df['label'] = le.fit_transform(df['unspsc_segment'])

train_df, test_df = train_test_split(
    df[['text', 'unspsc_segment', 'label']],
    test_size=0.20,
    stratify=df['label'],
    random_state=42
)

print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

train_df.to_parquet(f'{PROJ}/data/train.parquet', index=False)
test_df.to_parquet(f'{PROJ}/data/test.parquet', index=False)
joblib.dump(le, f'{PROJ}/models/label_encoder.pkl')

print('Saved: data/train.parquet, data/test.parquet, models/label_encoder.pkl')